In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Deteksi otomatis root project, supaya tidak tergantung dari mana notebook dibuka
current = Path(os.getcwd())
if (current / "data").exists():
    BASE_DIR = current
elif (current.parent / "data").exists():
    BASE_DIR = current.parent
else:
    raise FileNotFoundError("Folder 'data' tidak ditemukan, cek ulang lokasi notebook")

RAW_PATH = BASE_DIR / "data" / "raw" / "Pelayanan Kepuasan - Mahasiswa.xlsx"
OUTPUT_DIR = BASE_DIR / "data" / "clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR   :", BASE_DIR)
print("RAW_PATH   :", RAW_PATH, "| ada?", RAW_PATH.exists())
print("OUTPUT_DIR :", OUTPUT_DIR, "| ada?", OUTPUT_DIR.exists())

In [ ]:
# Kolom PII yang harus dibuang
KOLOM_PII = ["Nomor Gopay"]

# Kolom profil/identitas responden
KOLOM_PROFIL = ["Timestamp", "Program Studi", "Semester"]

# Kata kunci untuk mendeteksi kolom likert klinik (bersyarat)
KATA_KUNCI_KLINIK = "klinik"

# Kolom teks/open-ended - sesuaikan dengan redaksi persis di form
KOLOM_TEKS = [
    "Menurut Anda, layanan atau fasilitas apa yang perlu diperbaiki di kampus? Jelaskan perbaikan yang Anda harapkan. (Opsional)"
]

In [ ]:
def cek_file(path):
    if not path.exists():
        print(f"File TIDAK ditemukan di path: {path}")
        folder_raw = path.parent
        if folder_raw.exists():
            print("Isi folder raw saat ini:")
            for nama_file in os.listdir(folder_raw):
                print(f"  - {nama_file}")
        else:
            print(f"  Folder juga tidak ditemukan: {folder_raw}")
        return False
    print(f"File ditemukan: {path}")
    return True

if cek_file(RAW_PATH):
    df_raw = pd.read_excel(RAW_PATH)
    print(f"\nJumlah baris : {df_raw.shape[0]}")
    print(f"Jumlah kolom : {df_raw.shape[1]}")

In [ ]:
df = df_raw.copy()
df.insert(0, "response_id", ["MHS_" + str(i + 1).zfill(4) for i in range(len(df))])

kolom_pii_ada = [k for k in KOLOM_PII if k in df.columns]
df = df.drop(columns=kolom_pii_ada)
print(f"Kolom PII dibuang: {kolom_pii_ada}")

In [ ]:
semua_kolom = df.columns.tolist()

# Kolom likert = kolom yang mengandung tanda kurung siku "[...]"
kolom_likert_semua = [k for k in semua_kolom if "[" in k and "]" in k]

# Pisahkan klinik dari yang wajib
kolom_klinik = [k for k in kolom_likert_semua if KATA_KUNCI_KLINIK.lower() in k.lower()]
kolom_likert_utama = [k for k in kolom_likert_semua if k not in kolom_klinik]

print(f"Kolom likert utama : {len(kolom_likert_utama)}")
print(f"Kolom likert klinik: {len(kolom_klinik)}")
print(f"Kolom teks terbuka : {len(KOLOM_TEKS)}")

In [ ]:
import re

def bersihkan_likert(df_input, kolom_likert):
    df_hasil = df_input[kolom_likert].copy()
    for kol in kolom_likert:
        # Ambil angka di awal string, misal "1 - Tidak Baik" -> "1"
        df_hasil[kol] = df_hasil[kol].astype(str).str.extract(r"^(\d+)")[0]
        df_hasil[kol] = df_hasil[kol].replace("nan", np.nan)  # kembalikan NaN yang sempat jadi teks "nan"
        df_hasil[kol] = pd.to_numeric(df_hasil[kol], errors="coerce")
    return df_hasil

In [ ]:
kolom_dipakai_utama = ["response_id"] + KOLOM_PROFIL + kolom_likert_utama
df_likert_utama = df[kolom_dipakai_utama].copy()
df_likert_utama[kolom_likert_utama] = bersihkan_likert(df_likert_utama, kolom_likert_utama)

df_likert_utama.head()

In [ ]:
if len(kolom_klinik) == 0:
    print("Tidak ada kolom klinik ditemukan, cek KATA_KUNCI_KLINIK di Cell 2")
    df_likert_klinik = pd.DataFrame()
else:
    kolom_dipakai_klinik = ["response_id"] + kolom_klinik
    df_likert_klinik = df[kolom_dipakai_klinik].copy()
    df_likert_klinik[kolom_klinik] = bersihkan_likert(df_likert_klinik, kolom_klinik)

    # Buang baris yang semua kolom kliniknya kosong (memang tidak mengisi)
    df_likert_klinik = df_likert_klinik.dropna(subset=kolom_klinik, how="all")

    print(f"Responden yang mengisi data klinik: {len(df_likert_klinik)} dari total {len(df)} baris")

df_likert_klinik.head()

In [ ]:
df_teks_list = []

for kolom in KOLOM_TEKS:
    temp = df[["response_id", kolom]].copy()
    temp = temp.rename(columns={kolom: "teks_komentar"})
    temp["jenis_pertanyaan"] = kolom
    temp = temp[["response_id", "jenis_pertanyaan", "teks_komentar"]]
    df_teks_list.append(temp)

df_teks = pd.concat(df_teks_list, ignore_index=True)

# Buang baris kosong (tidak diisi responden)
df_teks = df_teks.dropna(subset=["teks_komentar"])
df_teks = df_teks[df_teks["teks_komentar"].str.strip() != ""]

print(f"Total jawaban teks terkumpul: {len(df_teks)} baris")
df_teks.head()

In [ ]:
df_likert_utama.to_csv(OUTPUT_DIR / "likert_mahasiswa_utama_clean.csv", index=False)
df_likert_klinik.to_csv(OUTPUT_DIR / "likert_mahasiswa_klinik_clean.csv", index=False)
df_teks.to_csv(OUTPUT_DIR / "teks_mahasiswa_clean.csv", index=False)

print("Selesai. 3 file tersimpan di:", OUTPUT_DIR)